# 07 · OpenSim on the new strides, with a per-trial viewer

**Run order:** 01 → … → 06 → **07** → 08. Needs `Results/cohort_events.pkl` and the per-runner OpenSim
artifacts in `OpenSim/Out/` (all 30 exist; regenerate any missing one with
`PYTHONPATH=. python "(Deprecated)/sprint_opensim.py"`, ~30 min, OpenSim 4.5).

**What "re-run" means here.** Scale, IK and the muscle analysis run over the *whole trial*, so those artifacts
don't depend on stride windows at all — they are reused as-is. What was stuck on the old foot-centre windows is
the **summary**: which frames count as the five top-speed strides when excursions, lengthening velocities and H6
are computed. This notebook recomputes that summary with the `stride.py` touchdown events (the same windows
notebooks 04–06 use), writes a fresh `OpenSim/Out/mtu_summary.csv`, and builds **`OpenSim/viewer.html`** — a
per-trial viewer with the IK agreement curves, muscle–tendon length curves and excursion shares for every
runner.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import cohort as C
import sprint_pipeline as SP

sys.path.append(str(Path("(Deprecated)").resolve()))
import sprint_opensim as OS                      # read-only reuse: read_storage, compare_angles, muscle groups

data = C.load()
F, runs = data["features"], data["runs"]
OUT = Path("OpenSim/Out")
pids = sorted(F.index)
print(f"{len(pids)} runners, artifacts in {OUT}/")

30 runners, artifacts in OpenSim/Out/


## 1 · Re-summarise every runner on the stride.py windows

Per runner: load the whole-trial IK agreement (window-independent), then window the muscle–tendon length curves
with the new touchdown-to-touchdown strides and recompute excursion, peak lengthening velocity, H6 and the
whole-body excursion shares. The loop also collects the cycle curves the viewer shows.

In [2]:
rows, viewer = [], {}
for pid in pids:
    res = SP.clean_for_angles(pid)
    run = runs[pid]
    cmp_df, ik, pipe = OS.compare_angles(res, OUT / f"{pid}_ik.mot")
    cmp_df = cmp_df.set_index("joint")
    L = OS.read_storage(OUT / f"{pid}_mtu_MuscleAnalysis_Length.sto")
    assert abs(len(L) - res["n_frames"]) <= 2, (pid, len(L), res["n_frames"])
    last = min(len(L), len(ik), res["n_frames"]) - 1

    windows = [(a.touchdown, min(b.touchdown, last)) for a, _m, b in C.top_speed_strides(run)]
    cyc = lambda s: np.mean([SP.resample_to_cycle(np.asarray(s, float)[a:b + 1]) for a, b in windows], axis=0)
    dt = np.mean([b - a for a, b in windows]) / res["fs"] / 100          # s per 1 % of the stride

    row = {"runner_id": pid,
           "knee_R_r_osim": cmp_df.loc["knee_R", "pearson_r_after_known_conversion"],
           "hip_R_r_osim": cmp_df.loc["hip_R", "pearson_r_after_known_conversion"],
           "hip_R_offset_osim_deg": cmp_df.loc["hip_R", "offset_deg_converted_mean"]}
    exc, mtu_curves = {}, {}
    for m in [c for c in L.columns if c != "time"]:
        curve = cyc(L[m])
        exc[m] = float(np.ptp(curve))
        if m in OS.KEY_MTU:
            v = np.gradient(curve / curve.mean(), dt)                    # lengths/s (+ = lengthening)
            row[f"{m}_excursion_pct"] = 100 * np.ptp(curve) / curve.mean()
            row[f"{m}_peak_lengthen_Lps"] = float(v.max())
            mtu_curves[m] = np.round(100 * curve / curve.mean(), 2).tolist()
            if m == "bifemlh_r":
                row["bflh_lengthen_Lps"] = float(v[70:].max())           # H6: late swing, 70-100 %
    body = {g: sum(exc[f"{m}_r"] for m in ms if f"{m}_r" in exc) for g, ms in OS.MTU_GROUPS.items()}
    body["trunk"] = sum(exc[m] for m in OS.TRUNK if m in exc)
    total = sum(body.values())
    row |= {f"share_{g}_pct": 100 * v_ / total for g, v_ in body.items()}
    rows.append(row)

    knee_osim = -OS._unwrap_deg(ik["knee_angle_r"].to_numpy()[:last + 1])   # documented sign conversion
    hip_osim = OS._unwrap_deg(ik["hip_flexion_r"].to_numpy()[:last + 1])
    viewer[pid] = {"speed": float(F.top_speed_ms[pid]), "sex": F.sex[pid],
                   "knee_r": row["knee_R_r_osim"], "hip_r": row["hip_R_r_osim"],
                   "hip_off": row["hip_R_offset_osim_deg"], "bflh": row["bflh_lengthen_Lps"],
                   "knee_pipe": np.round(cyc(pipe["knee_R"]), 2).tolist(),
                   "knee_osim": np.round(cyc(knee_osim), 2).tolist(),
                   "hip_pipe": np.round(cyc(pipe["hip_R"]), 2).tolist(),
                   "hip_osim": np.round(cyc(hip_osim), 2).tolist(),
                   "mtu": mtu_curves,
                   "share": {g: round(100 * v_ / total, 1) for g, v_ in body.items()}}
    print(pid, "ok", flush=True)
new = pd.DataFrame(rows).set_index("runner_id")

SB061 ok


SB101 ok


SB102 ok


SB110 ok


SB111 ok


SB112 ok


... (20 outputs left out)

SB81 ok


SB82 ok


SB91 ok


SB92 ok


## 2 · IK RMS carries over, QA, and the CSV

The per-frame IK marker-error file is overwritten by each run, so the per-runner RMS comes from the previous
summary (it is a whole-trial quantity — windows cannot change it). The old CSV is kept as
`mtu_summary_oldstrides.csv`.

In [3]:
backup = OUT / "mtu_summary_oldstrides.csv"                     # the true old-windows CSV, once it exists
old = pd.read_csv(backup if backup.exists() else OUT / "mtu_summary.csv").set_index("runner_id")
new.insert(0, "ik_rms_mm", old.ik_rms_mm)
qa = (new.knee_R_r_osim >= 0.95) & (new.hip_R_r_osim >= 0.95) & (new.ik_rms_mm <= 35)
for pid in pids:
    viewer[pid]["ik_rms_mm"] = float(new.ik_rms_mm[pid])
    viewer[pid]["qa"] = bool(qa[pid])

if not backup.exists():
    (OUT / "mtu_summary.csv").rename(backup)
new.reset_index().to_csv(OUT / "mtu_summary.csv", index=False)
print(f"wrote {OUT / 'mtu_summary.csv'} ({len(new)} runners); QA passes {int(qa.sum())}/30; old CSV -> {backup.name}")

wrote OpenSim/Out/mtu_summary.csv (30 runners); QA passes 25/30; old CSV -> mtu_summary_oldstrides.csv


## 3 · Old windows vs new windows

Whole-trial quantities must be identical; the windowed quantities are what the re-run changes.

In [4]:
shared = ["knee_R_r_osim", "hip_R_r_osim", "bifemlh_r_excursion_pct", "bflh_lengthen_Lps"]
side = pd.DataFrame({"old": old.loc[new.index, shared].stack(), "new": new[shared].stack()}).unstack(1)
print("max |Δ knee r| =", float((old.knee_R_r_osim - new.knee_R_r_osim).abs().max()).__round__(5))
delta = (new.bflh_lengthen_Lps - old.bflh_lengthen_Lps)
print(f"H6 (bflh late-swing lengthening): mean Δ {delta.mean():+.3f} L/s, max |Δ| {delta.abs().max():.3f}, "
      f"r(old,new) = {np.corrcoef(old.bflh_lengthen_Lps, new.bflh_lengthen_Lps)[0,1]:.3f}")
side.round(3).head(8)

max |Δ knee r| = 0.0
H6 (bflh late-swing lengthening): mean Δ +0.025 L/s, max |Δ| 1.092, r(old,new) = 0.849


old                                       \
          knee_R_r_osim hip_R_r_osim bifemlh_r_excursion_pct   
runner_id                                                      
SB061             0.990        0.973                  25.621   
SB101             0.994        0.978                  23.088   
SB102             0.989        0.950                  24.894   
SB110             0.999        0.983                  21.143   
SB111             0.732        0.951                  20.086   
SB112             0.997        0.983                  20.518   
SB15              0.995        0.987                  25.757   
SB150             0.995        0.990                  23.388   

                                      new               \
          bflh_lengthen_Lps knee_R_r_osim hip_R_r_osim   
runner_id                                                
SB061                 0.681         0.990        0.973   
SB101                 1.857         0.994        0.978   
SB102                 2.315         0.989        0.950   
SB110                 1.057         0.999        0.983   
SB111                 1.346         0.732        0.951   
SB112                 0.839         0.997        0.983   
SB15                  1.004         0.995        0.987   
SB150                 0.997         0.995        0.990   

                                                     
          bifemlh_r_excursion_pct bflh_lengthen_Lps  
runner_id                                            
SB061                      25.818             0.549  
SB101                      23.048             1.796  
SB102                      24.924             2.269  
SB110                      20.265             1.010  
SB111                      18.254             1.110  
SB112                      19.927             0.783  
SB15                       25.666             0.734  
SB150                      22.792             1.086

## 4 · H6 on the new windows (same QA rule as notebook 04)

In [5]:
y = F.top_speed_ms.to_numpy()
male = (F.sex == "M").to_numpy(float)
x = new.bflh_lengthen_Lps.reindex(F.index).to_numpy()
m = qa.reindex(F.index).to_numpy() & np.isfinite(x)
r2_s = SP.loo_r2_fast(male[m], y[m])
r2_h = SP.loo_r2_fast(np.column_stack([male, x])[m], y[m])
print(f"H6, {m.sum()} QA-passing runners: sex {r2_s:.3f} -> sex + H6 {r2_h:.3f} (gain {r2_h - r2_s:+.3f})")

H6, 25 QA-passing runners: sex 0.716 -> sex + H6 0.694 (gain -0.022)


## 5 · The per-trial viewer

One self-contained HTML file, `OpenSim/viewer.html`: pick any runner (sorted fastest first, ←/→ to flip) and see
their IK agreement curves, the six key muscle–tendon length curves with late swing shaded, the excursion-share
bars and the QA chips. Serve it with the `opensim-viewer` entry in `.claude/launch.json` or any static server.

In [6]:

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>OpenSim per-trial viewer</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  :root { --bg:#12151a; --card:#1b2028; --ink:#e8eaed; --mut:#9aa3ad; --line:#2a313b;
          --fast:#5b9bd5; --slow:#d56b6b; --ok:#3fae72; --bad:#d56b6b; }
  * { box-sizing:border-box; }
  body { margin:0; background:var(--bg); color:var(--ink);
         font:14px/1.45 -apple-system, "Segoe UI", Helvetica, Arial, sans-serif; }
  header { display:flex; flex-wrap:wrap; gap:12px; align-items:center;
           padding:14px 20px; border-bottom:1px solid var(--line); }
  h1 { font-size:16px; margin:0 12px 0 0; font-weight:600; }
  select,button { background:var(--card); color:var(--ink); border:1px solid var(--line);
                  border-radius:8px; padding:7px 10px; font-size:14px; }
  button { cursor:pointer; } button:hover { border-color:var(--mut); }
  .chips { display:flex; flex-wrap:wrap; gap:8px; padding:12px 20px 0; }
  .chip { background:var(--card); border:1px solid var(--line); border-radius:8px;
          padding:6px 12px; font-size:13px; }
  .chip b { font-size:15px; } .chip span { color:var(--mut); margin-right:6px; }
  .ok { color:var(--ok); } .bad { color:var(--bad); }
  main { display:grid; grid-template-columns:1fr 1fr; gap:14px; padding:14px 20px 28px; }
  .card { background:var(--card); border:1px solid var(--line); border-radius:12px; padding:10px; }
  .card h2 { font-size:13px; font-weight:600; color:var(--mut); margin:2px 6px 6px; }
  #ik, #mtu { height:380px; } #share { height:280px; }
  .wide { grid-column:1 / -1; }
  footer { color:var(--mut); font-size:12px; padding:0 20px 24px; max-width:1100px; }
  @media (max-width:900px) { main { grid-template-columns:1fr; } }
</style></head><body>
<header>
  <h1>OpenSim per-trial viewer — 60 m cohort</h1>
  <button id="prev">&#8592;</button>
  <select id="pick"></select>
  <button id="next">&#8594;</button>
  <span style="color:var(--mut);font-size:12px">strides = stride.py events &middot; kinematic only (no mass, no GRF)</span>
</header>
<div class="chips" id="chips"></div>
<main>
  <div class="card"><h2>IK agreement — knee_R (sign-converted) and hip_R, over the top-speed stride</h2><div id="ik"></div></div>
  <div class="card"><h2>Muscle–tendon length, % of trial mean (late swing shaded)</h2><div id="mtu"></div></div>
  <div class="card wide"><h2>Share of whole-body MTU excursion over the stride</h2><div id="share"></div></div>
</main>
<footer>Scale/IK/MuscleAnalysis run over the whole trial (gait2392, virtual Xsens landmarks); only the stride windows
summarised here come from the stride.py contact events. Hip shows a known ~21&deg; offset (trunk- vs pelvis-referenced).
H6 = peak biceps femoris LH lengthening speed in late swing (70&ndash;100&nbsp;%), in lengths/s. QA: knee r &ge; 0.95,
hip r &ge; 0.95, IK RMS &le; 35 mm.</footer>
<script>
const DATA = __DATA__;
const IDS = Object.keys(DATA).sort((a, b) => DATA[b].speed - DATA[a].speed);
const pick = document.getElementById("pick");
IDS.forEach(id => { const o = document.createElement("option"); o.value = id;
  o.textContent = `${id} — ${DATA[id].speed.toFixed(2)} m/s (${DATA[id].sex})`; pick.appendChild(o); });
const PCT = Array.from({length: 101}, (_, i) => i);
const L = { margin:{l:48, r:12, t:8, b:36}, paper_bgcolor:"#1b2028", plot_bgcolor:"#1b2028",
  font:{color:"#e8eaed", size:12}, xaxis:{title:"% of stride", gridcolor:"#2a313b", zeroline:false},
  yaxis:{gridcolor:"#2a313b", zeroline:false}, legend:{orientation:"h", y:1.14}, showlegend:true };
const CFG = {displayModeBar:false, responsive:true};
const MCOL = {bifemlh_r:"#d56b6b", rect_fem_r:"#5b9bd5", glut_max1_r:"#e0a84c",
              iliacus_r:"#9c7bd5", med_gas_r:"#3fae72", soleus_r:"#5bc8d5"};
function chip(label, val, cls) { return `<div class="chip"><span>${label}</span><b class="${cls || ""}">${val}</b></div>`; }
function show(id) {
  const d = DATA[id]; pick.value = id;
  document.getElementById("chips").innerHTML =
    chip("top speed", d.speed.toFixed(2) + " m/s") + chip("sex", d.sex) +
    chip("IK RMS", d.ik_rms_mm.toFixed(1) + " mm", d.qa ? "ok" : "bad") +
    chip("knee r", d.knee_r.toFixed(3), d.knee_r >= 0.95 ? "ok" : "bad") +
    chip("hip r", d.hip_r.toFixed(3), d.hip_r >= 0.95 ? "ok" : "bad") +
    chip("hip offset", d.hip_off.toFixed(1) + "°") +
    chip("QA", d.qa ? "pass" : "fail", d.qa ? "ok" : "bad") +
    chip("H6 late-swing BFlh", d.bflh.toFixed(2) + " L/s");
  Plotly.react("ik", [
    {x:PCT, y:d.knee_pipe, name:"knee pipeline", line:{color:"#5b9bd5", width:2}},
    {x:PCT, y:d.knee_osim, name:"knee OpenSim", line:{color:"#5b9bd5", width:2, dash:"dot"}},
    {x:PCT, y:d.hip_pipe, name:"hip pipeline", line:{color:"#d56b6b", width:2}},
    {x:PCT, y:d.hip_osim, name:"hip OpenSim", line:{color:"#d56b6b", width:2, dash:"dot"}},
  ], {...L, yaxis:{...L.yaxis, title:"degrees"}}, CFG);
  const mtu = Object.entries(d.mtu).map(([m, ys]) =>
    ({x:PCT, y:ys, name:m.replace("_r", ""), line:{color:MCOL[m], width:2}}));
  Plotly.react("mtu", mtu, {...L,
    yaxis:{...L.yaxis, title:"% of mean length"},
    shapes:[{type:"rect", x0:70, x1:100, y0:0, y1:1, yref:"paper",
             fillcolor:"#9aa3ad", opacity:0.12, line:{width:0}}]}, CFG);
  const groups = Object.keys(d.share);
  Plotly.react("share", [{type:"bar", orientation:"h", y:groups, x:groups.map(g => d.share[g]),
    marker:{color:"#5b9bd5"}}], {...L, showlegend:false,
    xaxis:{...L.xaxis, title:"% of total MTU excursion"}, yaxis:{...L.yaxis, automargin:true}}, CFG);
}
pick.onchange = () => show(pick.value);
document.getElementById("prev").onclick = () => show(IDS[(IDS.indexOf(pick.value) + IDS.length - 1) % IDS.length]);
document.getElementById("next").onclick = () => show(IDS[(IDS.indexOf(pick.value) + 1) % IDS.length]);
show(IDS[0]);
</script></body></html>"""

out = Path("OpenSim/viewer.html")
out.write_text(HTML_TEMPLATE.replace("__DATA__", json.dumps(viewer, separators=(",", ":"))))
print(f"wrote {out} ({out.stat().st_size / 1e6:.2f} MB, {len(viewer)} runners)")

wrote OpenSim/viewer.html (0.21 MB, 30 runners)


## 6 · Reading the results

* The whole-trial audit is untouched by construction (knee r identical to 5 decimals): the re-run changes
  *which frames are summarised*, not the simulation.
* H6 moves runner-by-runner with the corrected windows but stays what it was in substance: no held-out gain
  over sex. The windowed excursions in the new CSV are now consistent with every other notebook.
* The viewer is the per-trial microscope: QA failures are visible as IK disagreement at a glance.

In [7]:
assert len(new) == 30 and (OUT / "mtu_summary.csv").exists() and backup.exists()
assert (old.knee_R_r_osim - new.knee_R_r_osim).abs().max() < 1e-3      # whole-trial quantities unchanged
assert Path("OpenSim/viewer.html").exists() and Path("OpenSim/viewer.html").stat().st_size < 2e6
assert int(qa.sum()) >= 20
print("opensim re-run checks passed")

opensim re-run checks passed
